# 01 · Dataset overview
**Goal:** identify the cohort, exposure and gaps before looking at signal behaviour.

In [10]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts from the repository root or from nbs.
HELPERS = Path.cwd() if (Path.cwd() / "fleetguard_eda.py").exists() else Path.cwd() / "nbs"
sys.path.insert(0, str(HELPERS))
import fleetguard_eda as eda
import panel as pn
import plotly.express as px

eda.setup()
ROOT = eda.project_root()
SOURCE = Path(os.environ.get("FLEETGUARD_EDA_SOURCE", ROOT / "data/generated/portfolio-v1"))
FEATURES = Path(os.environ.get("FLEETGUARD_EDA_FEATURES", ROOT / "data/features/portfolio-v1"))
SPLIT = "train"  # change to validation to check findings; test is blocked for exploration
ws = eda.Workspace(SOURCE, FEATURES)
pn.pane.Markdown(f"**Source:** `{SOURCE}`  ·  **Features:** `{FEATURES}`  ·  **Explore:** `{SPLIT}`\n\nMaps need internet. Linked controls need a running, trusted notebook kernel.")

Markdown(str)

In [11]:
eda.banner("01 · Know the dataset", "Coverage → identities → reporting gaps → journeys")

HTML(str, sizing_mode='stretch_width')

### 1 · Confirm the source
These checks scan both saved datasets. Test is used only for inventory/coverage.

In [3]:
CHECK_SAVED_FILES = True
checks = ws.validate() if CHECK_SAVED_FILES else {"note": "Validation skipped; run CLI validators first."}
checks

{'portfolio': 'passed', 'features': 'passed'}

### 2 · Coverage and unique identities
Open each tab. Counts use complete parent keys, not just inner/outer or left/right.

In [4]:
eda.overview_views(ws)

Tabs
    [0] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [1] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [2] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [3] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=        split  ...)
    [4] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=         split  ...)

### Component reporting frequency
Actual received-report counts use full component keys. Train/validation only.

In [5]:
eda.identity_frequency(ws, SPLIT)

Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=        asset_id  ...)

### 3 · When did reports disappear?
White cells outside a duty are not outages. Completeness uses expected report slots.

In [6]:
eda.pane(eda.completeness_view(ws, SPLIT))

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

### 4 · Explore one wagon on the map
Markers are sampled; heat layers use all received wagon reports. No wheel duplication.

In [13]:
wagons = ws.load("wagon", SPLIT)
asset_picker = pn.widgets.Select(name="Wagon", options=["All", *sorted(wagons.asset_id.unique())])
def route_view(asset):
    selected = eda.choose(wagons, asset=asset)
    return pn.pane.HTML(eda.folium_map(selected)._repr_html_(), height=620, sizing_mode="stretch_width")
pn.Column(asset_picker, pn.bind(route_view, asset_picker))

Column
    [0] Select(label='Wagon', name='Wagon', options=['All', 'FG-WGN-0002', ...], value='All')
    [1] ParamFunction(function, _pane=HTML, defer_load=False)

### 5 · Plotly geographic hover
Interactive positions with speed and timestamp; coarse simulated coordinates.

In [8]:
eda.pane(eda.plotly_map(wagons))

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

### 6 · Outage replay
Expected missing slots are synthetic truth, not observations of the controller during silence.

In [9]:
eda.pane(eda.outage_view(ws, SPLIT))

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

### 7 · Record decisions
- Which identities/routes have limited exposure?
- Which wagons have partial/silent minutes?
- Do the 1:2:4:8 component counts reconcile?

**Next session:** healthy signal behaviour. Keep test signal values unopened.